# Inital EDA ([M3-05] First look at the test-pull data)

In [3]:
!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.2 MB/s  0:00:04 eta 0:00:010:00:01

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# 1. Locate and load the parquet file in data/processed/
processed_dir = Path("../data/processed")
parquet_files = list(processed_dir.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No parquet files found in {processed_dir.absolute()}"
    )

file_path = parquet_files[0]

# Load using pyarrow directly to avoid the pandas/pyarrow patch conflict
print(f"--> Loading dataset: {file_path}")
table = pq.read_table(file_path)
df = table.to_pandas()

print(f"\n{'='*50}")
print(f" DATASET OVERVIEW ")
print(f"{'='*50}")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\nColumns present:")
for col in df.columns:
    print(f" - {col} ({df[col].dtype})")


print(f"\n{'='*50}")
print(f" 1. MISSING & EMPTY FIELDS CHECK ")
print(f"{'='*50}")
missing_counts = df.isnull().sum()
print("Null value counts per column:")
print(missing_counts[missing_counts > 0])

text_cols = [
    col
    for col in ["body", "title", "question_body", "text"]
    if col in df.columns
]
for col in text_cols:
    empty_strings = (df[col].astype(str).str.strip() == "").sum()
    print(f"Empty/blank strings in '{col}': {empty_strings}")


print(f"\n{'='*50}")
print(
    f" 2. UNIX TIMESTAMP CONVERSION & DATE RANGE CHECK "
)  # [M2-06 / M3-05 Check]
print(f"{'='*50}")

# Identify likely Unix timestamp columns (numeric columns with names containing date, time, creation, etc.)
date_cols = [
    col
    for col in df.columns
    if any(
        term in col.lower()
        for term in ["date", "time", "creation", "creation_date"]
    )
    or pd.api.types.is_numeric_dtype(df[col])
]

# Filter down to numeric columns that likely represent epoch timestamps (e.g. values > 1 billion)
timestamp_cols = []
for col in date_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        non_nulls = df[col].dropna()
        if not non_nulls.empty and non_nulls.median() > 1e9:  # Epoch time > ~2001
            timestamp_cols.append(col)

if timestamp_cols:
    for col in timestamp_cols:
        print(f"\nConverting Unix timestamp column: '{col}' to UTC datetime...")
        # Convert Unix timestamp (seconds) to UTC datetime
        dt_series = pd.to_datetime(df[col], unit="s", errors="coerce")

        print(f" - Min Date: {dt_series.min()}")
        print(f" - Max Date: {dt_series.max()}")
        print(f" - NaT (Invalid Conversion) Count: {dt_series.isnull().sum()}")

        # Check against Stack Overflow launch (Sept 12, 2008) and future dates
        too_early = (dt_series < "2008-09-01").sum()
        in_future = (dt_series > pd.Timestamp.now()).sum()
        print(
            f" - Records before Stack Overflow launch (Sept 2008): {too_early}"
        )
        print(f" - Records with future dates: {in_future}")
else:
    print(
        "No columns found matching timestamp criteria. Check your column names manually if needed."
    )


print(f"\n{'='*50}")
print(f" 3. EXTREME VALUES & BODY LENGTHS ")
print(f"{'='*50}")
for col in text_cols:
    lengths = df[col].astype(str).str.len()
    print(f"Body length stats for '{col}':")
    print(lengths.describe(percentiles=[0.5, 0.90, 0.95, 0.99]))

    print(f"\nTop 3 longest indices for '{col}':")
    longest_indices = lengths.nlargest(3).index
    for idx in longest_indices:
        snippet = (
            str(df.loc[idx, col])[:200] + "..."
            if len(str(df.loc[idx, col])) > 200
            else str(df.loc[idx, col])
        )
        print(f" - Index {idx} (Length: {lengths[idx]} chars):")
        print(f"   Snippet: {snippet}\n")


print(f"\n{'='*50}")
print(f" 4. BASIC DISTRIBUTIONS ")
print(f"{'='*50}")
numeric_cols = df.select_dtypes(include=["number"]).columns
if len(numeric_cols) > 0:
    print(df[numeric_cols].describe())
else:
    print("No numeric columns found.")

if "tags" in df.columns:
    print("\nTop 10 most common tags/tag combinations:")
    print(df["tags"].value_counts().head(10))

--> Loading dataset: ../data/processed/test_pull_python_sample.parquet

 DATASET OVERVIEW 
Total Rows: 200
Total Columns: 21

Columns present:
 - tags (object)
 - is_answered (bool)
 - view_count (int64)
 - accepted_answer_id (float64)
 - answer_count (int64)
 - score (int64)
 - last_activity_date (int64)
 - creation_date (int64)
 - last_edit_date (float64)
 - question_id (int64)
 - content_license (object)
 - link (object)
 - title (object)
 - body (object)
 - closed_date (float64)
 - closed_reason (object)
 - protected_date (float64)
 - posted_by_collectives (object)
 - migrated_from (object)
 - sampling_weight (float64)
 - owner_user_hash (object)

 1. MISSING & EMPTY FIELDS CHECK 
Null value counts per column:
accepted_answer_id       125
last_edit_date            71
content_license           18
closed_date              182
closed_reason            182
protected_date           200
posted_by_collectives    200
migrated_from            200
owner_user_hash            1
dtype: int64
Em

In [13]:
print(df[df["owner_user_hash"].isna()]["link"])

129    https://stackoverflow.com/questions/78130426/t...
Name: link, dtype: object


### Takeaways from Test-Pull EDA ([M3-05])

#### 1. Dataset Overview & Volume
- **Scale:** Successfully loaded 200 rows and 21 columns from the test-pull parquet dataset.
- **Scope:** Confirmed data spans a 1-month window of Python questions (`creation_date` ranges from March 2024).

#### 2. Missing & Empty Fields Audit
- **Expected Missingness:** `accepted_answer_id` has 125 missing values, which aligns with questions that lack an accepted answer (mapped to target 0 in `M2-08`). `closed_date` and `closed_reason` have 182 missing (unclosed questions).
- **Fully Unpopulated Fields:** `protected_date`, `posted_by_collectives`, and `migrated_from` are entirely null (0 counts) across this sample.
- **Text Integrity:** Zero empty or blank strings found in the `body` or `title` fields. Only 1 missing `owner_user_hash`.

#### 3. Timestamp Conversion & Date Checks
- **Conversion Success:** Unix timestamp columns (`creation_date`, `last_activity_date`, `last_edit_date`, `closed_date`) converted cleanly to UTC datetimes with zero conversion errors (`NaT`).
- **Temporal Bounds:** No dates fall prior to Stack Overflow's launch (September 2008) or appear invalid. `last_activity_date` and `last_edit_date` correctly capture ongoing activity up to March 2026.

#### 4. Body Lengths & Extreme Values
- **Body Lengths:** Median length is 1,537.5 characters, with a 99th percentile of 15,575 characters and a maximum of 27,817 characters. 
- **Outlier Check:** Inspection of the longest bodies shows legitimate, detailed technical questions (involving PLC/OPCUA, CNNs, and file manipulation) rather than spam or injection junk.

#### 5. Recommendations for Member 2 (Cleaning Pipeline)
- **Target Logic (`M2-08`):** Ensure missing `accepted_answer_id` entries are correctly handled as target 0 (unaccepted/unresolved) rather than dropped as missing values.
- **Missing owner_user_hash:** Missing owner_user_hash value for one entry to be investigated and handled properly